# Developing an environment

These examples use this checkout of New RecSim. From the repository root, create a virtual environment and run `python -m pip install -e ".[rl,docs]"`. Select that Python environment as your notebook kernel. All examples run on CPU and write only to temporary directories. The simulator itself does not require PyTorch.

Copyright 2019 The RecSim Authors. Licensed under the Apache License, Version 2.0.

## Documents and samplers

A document owns an ID and observable features. A sampler owns its random generator. The raw simulator accepts arbitrary document IDs; the wrapper exposes fixed candidate slots, so resampling does not change the observation space.

In [ ]:
import numpy as np
from gymnasium import spaces
from recsim import document, user
from recsim.simulator.environment import Environment
from recsim.simulator.recsim_gym import RecSimGymEnv


class TopicDocument(document.AbstractDocument):
    def __init__(self, doc_id, quality):
        super().__init__(doc_id)
        self.quality = quality

    def create_observation(self):
        return np.array([self.quality], dtype=np.float32)

    @classmethod
    def observation_space(cls):
        return spaces.Box(0.0, 1.0, shape=(1,), dtype=np.float32)


class TopicSampler(document.AbstractDocumentSampler):
    def __init__(self, seed=0):
        super().__init__(TopicDocument, seed=seed)

    def sample_document(self):
        result = TopicDocument(self._doc_count, self._rng.uniform())
        self._doc_count += 1
        return result

## User state and responses

The observed preference is a float32 vector. The response declares a scalar click field. The base user sampler already provides a local RNG, which the simulator reseeds independently from document and response randomness.

In [ ]:
class TopicUserState(user.AbstractUserState):
    def __init__(self, preference):
        self.preference = preference
        self.remaining = 8

    def create_observation(self):
        return np.array([self.preference], dtype=np.float32)

    @staticmethod
    def observation_space():
        return spaces.Box(0.0, 1.0, shape=(1,), dtype=np.float32)


class TopicUserSampler(user.AbstractUserSampler):
    def __init__(self, seed=0):
        super().__init__(TopicUserState, seed=seed)

    def sample_user(self):
        return TopicUserState(self._rng.uniform())


class ClickResponse(user.AbstractResponse):
    def __init__(self, clicked=False):
        self.clicked = clicked

    @staticmethod
    def response_space():
        return spaces.Dict({"click": spaces.Discrete(2)})

    def create_observation(self):
        return {"click": int(self.clicked)}

## Response and transition model

Here the first item is clicked with probability preference × quality. A session lasts eight recommendations. This intentionally simple model isolates the contracts required to develop a richer simulator.

In [ ]:
class TopicUserModel(user.AbstractUserModel):
    def __init__(self, slate_size, seed=0):
        super().__init__(ClickResponse, TopicUserSampler(seed), slate_size)

    def simulate_response(self, documents):
        responses = [ClickResponse() for _ in documents]
        probability = self._user_state.preference * documents[0].quality
        responses[0].clicked = self._rng.random() < probability
        return responses

    def update_state(self, documents, responses):
        self._user_state.remaining -= 1

    def is_terminal(self):
        return self._user_state.remaining == 0


raw = Environment(TopicUserModel(2), TopicSampler(), num_candidates=4, slate_size=2)
env = RecSimGymEnv(raw, lambda responses: sum(r.clicked for r in responses), seed=42)

## Validate and run

Always check both reset and step observations against the declared space. `check_env` additionally checks seeded repeatability. Return detached arrays so a later state transition cannot overwrite a stored observation.

In [ ]:
from gymnasium.utils.env_checker import check_env

check_env(env, skip_render_check=True)
obs, _ = env.reset(seed=42)
steps = 0
while True:
    obs, reward, terminated, truncated, info = env.step([0, 1])
    steps += 1
    assert env.observation_space.contains(obs)
    if terminated or truncated:
        break
assert steps == 8 and terminated and not truncated
env.close()
print("Validated custom environment:", steps, "steps")